# 📊 CVM Data Extraction - Elite Tickers B3

Este notebook implementa a extração otimizada de dados financeiros (DRE) diretamente do portal de dados abertos da CVM, focando estritamente em uma lista de elite de tickers da B3.

**Destaques:**
1. **Mapeamento Preciso:** Converte Tickers em Códigos CVM via arquivos FCA.
2. **Big Data Filter:** Filtra dados brutos de DRE desde 2015.
3. **Memória Otimizada:** Descarta empresas irrelevantes logo na leitura dos arquivos ZIP.

In [ ]:
import pandas as pd
import requests
import zipfile
import io
import os
import time
from datetime import datetime
from pathlib import Path

# Configurações de pastagem
PASTA_OUTPUT = Path("../data/raw")
PASTA_OUTPUT.mkdir(parents=True, exist_ok=True)

print("✅ Ambiente configurado.")

## 1. Definição do Universo Alvo
Lista de tickers de elite que desejamos monitorar.

In [ ]:
TICKERS_ALVO = [
    "PETR4","ITUB4","VALE3","BPAC11","ABEV3","WEGE3","BBDC3","AXIA3","ITSA4","BBAS3","VIVT3","SANB11","SBSP3",
    "B3SA3","RDOR3","SUZB3","BBSE3","EMBJ3","CPLE3","TIMS3","RENT3","CPFE3","CXSE3","EQTL3","PRIO3","RADL3",
    "NEOE3","ENEV3","GGBR3","EGIE3","CMIG4","VBBR3","PSSA3","MOTV3","CMIN3","RAIL3","UGPA3","MBRF3","ENGI11",
    "KLBN11","CSAN3","TOTS3","CSMG3","ISAE4","CGAS3","MULT3","BPAN4","REDE3","ALOS3","MRSA3B","LREN3","TAEE11",
    "EQPA3","CEGR3","HYPE3","GGPS3","SAPR11","CYRE3","CEEB3","GMAT3","AURE3","BNBR3","AZUL54","SMFT3","NATU3",
    "ALUP11","ASAI3","GOAU4","ENMT4","CURY3","CSNA3","FLRY3","ALPA4","BRAP4","USIM5","BRAV3","TTEN3","EKTR3",
    "CASN3","SLCE3","POMO4","BMEB4","IGTI3","DIRR3","SRNA3","MDIA3","BRSR6","UNIP6","ODPV3","VIVA3","BRKM5",
    "RAIZ4","MGLU3","ECOR3","ORVR3","COGN3","FRAS3","WHRL4","CBAV3","ABCB4","JHSF3","SMTO3","VULC3","HBSA3",
    "EQMA3B","CLSC4","MRVE3","SHUL4","AZZA3","HAPV3","SIMH3","BAZA3","BEEF3","IRBR3","LEVE3","MOVI3","DXCO3",
    "RIAA3","DASA3","CBEE3","VAMO3","GRND3","INTB3","PGMN3","EZTC3","CEAB3","RECV3","TEND3","MILS3","LAVV3",
    "LOGN3","COCE5","YDUQ3","FESA4","GEPA4","EMAE4","MDNE3","BEES3","BMGB4","PINE4","PLPL3","SBFG3","BHIA3",
    "AUAU3","TGMA3","TFCO4","ONCO3","LOGG3","BLAU3","CSED3","CAML3","PNVL3","RAPT4","RANI3","JSLG3","CEBR3",
    "BSLI3","AGRO3","FIQE3","BMOB3","ARML3","MATD3","LWSA3","EUCA3","VTRU3","OPCT3","ANIM3","LIGT3","TRIS3",
    "VLID3","TUPY3","SEQL3","EVEN3","DESK3","MYPK3","KEPL3","WIZC3","SEER3","ZAMP3","OFSA3","BRST3","TELB3",
    "PCAR3","OBTC3","PRNR3","SOJA3","GPAR3","PFRM3","BGIP4","CVCB3","ALPK3","FRIO3","MOSI3","MLAS3","AMER3",
    "DEXP3","JALL3","BIOM3","LAND3","WLMM3","SCAR3","CSUD3","TASA3","MELK3","PATI3","ROMI3","AALR3","SYNE3",
    "PEAB3","ALLD3","CEED3","CGRA4","TKNO4","MERC4","VITT3","MTSA4","POSI3","QUAL3","CRPG5","TECN3","AMOB3",
    "RNEW4","AFLT3","VVEO3","AMAR3","PTBL3","RPAD3","VSTE3","DOHL4","LJQQ3","ESPA3","MTRE3","PTNT4","HBOR3",
    "CAMB3","CASH3","AMBP3","DMVF3","MEAL3","EALT4","HBRE3","MNP3"
]

## 2. Tradução Ticker -> Código CVM
Mapeamos os códigos numéricos da CVM necessários para baixar os balanços.

In [ ]:
def gerar_mapa_cvm_focado(tickers_alvo):
    """
    Busca os códigos da CVM de forma robusta, cruzando via CNPJ se necessário.
    """
    ano_referencia = datetime.now().year - 1
    print(f"🕵️‍♂️ Mapeando Códigos CVM para a sua lista de Elite (Ano Base: {ano_referencia})...")
    
    url_fca = f"https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/FCA/DADOS/fca_cia_aberta_{ano_referencia}.zip"
    
    try:
        resposta = requests.get(url_fca, timeout=60)
        if resposta.status_code != 200:
            print(f"❌ Erro ao baixar o arquivo FCA ({resposta.status_code}).")
            return {}
            
        arquivo_zip = zipfile.ZipFile(io.BytesIO(resposta.content))
        nome_ficheiro_acoes = f"fca_cia_aberta_valor_mobiliario_{ano_referencia}.csv"
        nome_ficheiro_geral = f"fca_cia_aberta_geral_{ano_referencia}.csv"
        
        dicionario_cvm_ticker = {}
        
        with arquivo_zip.open(nome_ficheiro_acoes) as f:
            df_fca = pd.read_csv(f, sep=';', encoding='ISO-8859-1')
            
            # Detecção inteligente de colunas
            cols = {c.upper(): c for c in df_fca.columns}
            col_ticker = cols.get('CODIGO_NEGOCIACAO', cols.get('COD_NEGOCIACAO'))
            col_cvm = cols.get('CODIGO_CVM', cols.get('CD_CVM'))
            
            # Se CD_CVM não estiver no arquivo de ações (comum em 2025), busca no arquivo geral via CNPJ
            if not col_cvm:
                print("   🔍 CD_CVM ausente no arquivo de ativos, realizando merge com arquivo geral...")
                with arquivo_zip.open(nome_ficheiro_geral) as f_g:
                    df_geral = pd.read_csv(f_g, sep=';', encoding='ISO-8859-1')
                    cols_g = {c.upper(): c for c in df_geral.columns}
                    col_cvm_g = cols_g.get('CODIGO_CVM', cols_g.get('CD_CVM'))
                    
                    df_cvm_map = df_geral[['CNPJ_Companhia', col_cvm_g]].drop_duplicates()
                    df_fca = pd.merge(df_fca, df_cvm_map, on='CNPJ_Companhia', how='left')
                    col_cvm = col_cvm_g

            df_fca = df_fca.dropna(subset=[col_ticker, col_cvm])
            grupos = df_fca.groupby(col_cvm)
            
            for cd_cvm, grupo in grupos:
                tickers = [str(t).strip() for t in grupo[col_ticker].unique()]
                for t in tickers:
                    if t in tickers_alvo:
                        dicionario_cvm_ticker[int(cd_cvm)] = t
                        break
                
        print(f"   ✅ SUCESSO! Mapeados {len(dicionario_cvm_ticker)} ativos da Elite.")
        return dicionario_cvm_ticker

    except Exception as e:
        print(f"   ❌ Falha no mapeamento: {e}")
        return {}

mapa_cvm = gerar_mapa_cvm_focado(TICKERS_ALFO)
if not mapa_cvm:
    print("💥 ERRO: Não foi possível mapear nenhum ticker. Verifique a conexão ou os arquivos da CVM.")

## 3. Extração dos Balanços (DRE Histórico)
Loop pelos anos solicitados capiturando Receita e Lucro.

In [ ]:
def extrair_balancos(mapa_cvm, ano_inicio=2015, ano_fim=2024):
    mapa_contas = {
        '3.01': '01_Receita_Liquida_R$',
        '3.03': '02_Lucro_Bruto_R$',
        '3.05': '03_EBIT_Operacional_R$',
        '3.06': '04_Resultado_Financeiro_R$',
        '3.09': '05_Lucro_Antes_Impostos_EBT_R$',
        '3.11': '06_Lucro_Liquido_R$'
    }
    contas_alvo = list(mapa_contas.keys())
    codigos_cvm_alvo = list(mapa_cvm.keys())
    lista_balancos = []

    for ano in range(ano_inicio, ano_fim + 1):
        print(f"🚀 Processando DRE Anual {ano}...")
        url = f"https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/dfp_cia_aberta_{ano}.zip"
        try:
            res = requests.get(url, timeout=40)
            if res.status_code == 200:
                zp = zipfile.ZipFile(io.BytesIO(res.content))
                fname = f"dfp_cia_aberta_DRE_con_{ano}.csv"
                if fname in zp.namelist():
                    with zp.open(fname) as f:
                        df = pd.read_csv(f, sep=';', encoding='ISO-8859-1')
                        # Filtro massivo de memória
                        df_f = df[df['CD_CVM'].isin(codigos_cvm_alvo)].copy()
                        df_f = df_f[df_f['ORDEM_EXERC'] == 'ÚLTIMO']
                        df_f = df_f[df_f['CD_CONTA'].isin(contas_alvo)]
                        df_f['Conta_Nome'] = df_f['CD_CONTA'].map(mapa_contas)
                        lista_balancos.append(df_f)
                        print(f"   ✅ {ano} extraído.")
                else:
                    print(f"   ⚠️ Arquivo DRE não encontrado para {ano}.")
        except Exception as e:
            print(f"   ❌ Erro em {ano}: {e}")
            
    return pd.concat(lista_balancos, ignore_index=True) if lista_balancos else pd.DataFrame()

if mapa_cvm:
    df_raw = extrair_balancos(mapa_cvm)
    print(f"✅ Total de {len(df_raw)} registros de balanços capturados.")
else:
    df_raw = pd.DataFrame()

## 4. Pivotagem e Exportação Final
Transformamos as linhas em colunas e salvamos o CSV oficial.

In [ ]:
if not df_raw.empty:
    print("⚙️ Refinando dataset final...")
    df_pivot = df_raw.pivot_table(
        index=['DENOM_CIA', 'CD_CVM', 'DT_FIM_EXERC'],
        columns='Conta_Nome',
        values='VL_CONTA',
        aggfunc='sum'
    ).reset_index()
    
    # Mapear Ticker Amigável
    df_pivot['Ticker_Alvo'] = df_pivot['CD_CVM'].map(mapa_cvm)
    df_pivot['Ano_Exercicio'] = pd.to_datetime(df_pivot['DT_FIM_EXERC']).dt.year
    
    # Ajustar unidades monetárias
    cols_dinheiro = [c for c in df_pivot.columns if 'R$' in c]
    for col in cols_dinheiro: df_pivot[col] = df_pivot[col] * 1000
    
    # Ordenar e Limpar
    df_final = df_pivot.sort_values(['Ticker_Alvo', 'Ano_Exercicio'])
    df_final = df_final.dropna(subset=['Ticker_Alvo'])
    
    caminho = PASTA_OUTPUT / "05_CVM_Historico_Focado.csv"
    df_final.to_csv(caminho, sep=';', decimal=',', index=False, encoding='utf-8-sig')
    
    print(f"🚀 SUCESSO TOTAL! Arquivo salvo em: {caminho}")
    display(df_final.head(15))
else:
    print("❌ Falha: Nenhum dado para processar.")